# B2.11 · Remediation engineering — proven in a sandbox before the merge request

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline, Before and After Deploy**  ·  *AI for Security*

Builds on **[B2.10 · Severity calibration, triaging and reporting](https://spbreed.github.io/cyber-commons/lessons/B2.10.html)**.

| | |
|---|---|
| Tools used | Semgrep OSS, pytest, GLM-4.6, Kimi K2, Claude Sonnet 5 |

## What this lesson is

**What it covers.** Validate four candidate patches on three axes and show which of them only made the scanner green.

**Why a security engineer needs it.** A patch that silences the scanner is indistinguishable from a patch that fixes the bug. The control it builds is: stage 14: generate the fix, re-run the exploit against the patched build, and require a regression test.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A patch that passes the tests and changes the behaviour is not a fix, it is a second incident with a pull request attached. Remediation is the stage where the pipeline stops finding things and starts touching them.

> **At CyberTravels.** The Coding Agent's fix must not break booking behaviour. A patch that passes the tests and changes what travellers experience is a second incident with a pull request attached. R8.

## 2 · The framework

```
   patch                    what has to be true
   +----------------+       +-----------------------------+
   | fixes the bug  |  and  | behaviour unchanged         |
   |                |       | tests still pass            |
   |                |       | reviewer can follow the why |
   +----------------+       +-----------------------------+

   a patch that passes the tests and changes the behaviour is
   a second incident with a pull request attached
```

**Stage 14 — Remediation engineering.** Generate the fix, prove it, and only
then ask anybody to look at it.

This stage comes after triage on purpose. Remediation is the first stage that
*touches* the codebase, and touching it in the wrong order is expensive: fixing
by rule severity means fixing the finding at the top of a queue that B2.10 has
just told you is sorted wrongly. Calibrate first, then remediate, and the fix
you write first is the one that matters most.

A model that finds bugs is useful. A model that fixes them is only useful if you
can tell a real fix from a plausible one, and plausible is exactly what language
models are optimised to produce.

There are three ways to make a finding stop firing:

1. **Fix the vulnerability** — behaviour preserved, bug gone.
2. **Remove the code** — finding gone, so is the feature.
3. **Evade the detector** — rewrite until the pattern misses.

All three make the scanner green, and an autonomous loop optimising for a green
scan will find options 2 and 3 on its own because they are cheaper.

### The fix does not go straight to a merge request

The pipeline has two things a static workflow does not, and both were built
earlier in this chapter: Phase 4 produced a **working exploit**, and B2.6 built
a **replica** to run it against. So a patch has somewhere to be proven before a
reviewer's time is spent on it, and the promotion path is three environments
with a different question at each:

| where | the question | what it catches |
|---|---|---|
| **sandbox replica** | does the exploit still work? | the patch that only moved the pattern |
| **QA** | does the application still do what it did? | the patch that closed the bug by breaking the feature |
| **merge request to `master`** | is this the mechanism we want? | the design question no gate can answer |

The order is not arbitrary. The exploit re-run is first because it is the only
acceptance test that **cannot be satisfied by editing the code around the
detector**, and it is the one that gets skipped, always with the same
justification: the fix is obvious.

One run that sounds redundant and is not: the exploit must also be re-run
against the **unpatched** build in the same harness. Without it you cannot tell
"the fix worked" from "the probe broke", and a probe that has quietly stopped
asserting reports every candidate as a success.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · The stage, as a skill

Several candidate patches make the scanner green; one of them is a fix. The skill runs all three gates — behaviour unchanged, exploit blocked, and proof of fix against the old build — and reports which gate each rejected candidate died at.

### The skill — [`skills/appsec/patch-validation-harness/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/patch-validation-harness/SKILL.md)

```yaml
name: patch-validation-harness
description: >-
  Accept a proposed fix only when three things hold — behaviour unchanged, the
  exploit no longer works, and the fix proved against the build that was
  vulnerable. Use when an agent proposes a patch, or when a scanner going green
  is being read as a fix.
allowed-tools: Read, Grep, Glob, Bash
```

# A green scanner is not a fixed bug

Several candidate patches will make the scanner green. Some of them change
behaviour, some of them leave the bug exploitable, and one of them does neither.
Telling them apart needs three gates, and the third is the one that is usually
missing: proof against the **old** build, so "the exploit stops working" is a
statement about the patch rather than about the environment.

## When to use this

Every proposed remediation, whether authored by a person or an agent, and
especially when the evidence offered is that the scanner no longer fires.

## Procedure

**1 — Establish the baseline on the vulnerable build.** The behaviour cases must
pass and the exploit must work. If the exploit does not work here, you are about
to validate a patch against a bug you have not reproduced.

**2 — Gate one: behaviour unchanged.** Run every behaviour case against the
patched build. A patch that fixes the defect and changes an answer is a
regression with a security justification.

**3 — Gate two: the exploit no longer works.** Against the patched build,
directly. Not "the scanner is quiet" — the scanner was one of the tools that
missed the defect class in the first place.

**4 — Gate three: proof of fix.** Run the exploit against the old build again,
after the patch is written, in the same harness. It must still work. This is
what excludes the environment having changed underneath the test.

**5 — Report per candidate and per gate.** A candidate rejected at gate one and
one rejected at gate two need different conversations with whoever wrote them.

## Output contract

```json
{
  "baseline": {"behaviour_pass": true, "exploit_works": true},
  "candidates": [{"id": "str", "scanner_green": true,
                  "behaviour_unchanged": true, "exploit_blocked": true,
                  "proof_of_fix": true, "verdict": "accepted|rejected", "rejected_at": "str|null"}],
  "accepted": ["str"]
}
```

## Failure modes

- **Accepting a green scanner.** Several wrong patches produce one.
- **Skipping the behaviour cases.** The most reliable way to block an exploit is
  to break the feature.
- **Omitting proof of fix.** Without it you cannot distinguish a working patch
  from a broken exploit.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/patch-validation-harness/scripts/patch_validation_harness.py
SCRIPT = "skills/appsec/patch-validation-harness/scripts/patch_validation_harness.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## 4 · From the sandbox to the merge request

Validation says a patch is good. It does not say the patch may be merged, and those are different decisions taken in different places. This skill runs the promotion path: the exploit re-run in the replica, the regression suite in QA, the finding diff in both directions, and only then a merge request against `master` with the evidence attached to it.

## 5 · The promotion path, as a skill

The same four candidates, walked through three environments. Watch what CI alone would have merged against what survives the gates — and then watch candidate D pass every one of them.

### The skill — [`skills/appsec/fix-promotion-gate/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/fix-promotion-gate/SKILL.md)

```yaml
name: fix-promotion-gate
description: >-
  Decide whether a candidate security patch may leave the sandbox, pass QA and
  become a merge request against the mainline. Use when an agent or a developer
  proposes a fix for a confirmed finding, when a green CI run is being treated
  as proof a vulnerability is closed, or when deciding what a security fix must
  prove before review.
allowed-tools: Read, Grep, Glob, Bash
```

# What must a fix prove, and where, before anyone is asked to review it

A green pipeline says the tests that exist still pass. For a security patch
that is the wrong question, because the tests that exist were written by people
who did not know about this bug — if they had, it would not be here.

There are three ways to make a finding stop firing and only one of them is a
fix: **fix the vulnerability**, **delete the feature**, or **evade the
detector**. All three turn CI green, and an autonomous loop optimising for
green finds the second and third on its own, because they are cheaper.

The pipeline has something a static workflow does not: Phase 4 already built a
**working exploit**, and B2.6 already built a **replica** to run it against. So
the fix has somewhere to be proven before anybody is asked to look at it, and
the promotion path is three environments with a different question at each:

| where | the question | what it catches |
|---|---|---|
| **sandbox replica** | does the exploit still work? | the patch that only moved the pattern |
| **QA** | does the application still do what it did? | the patch that closed the bug by breaking the feature |
| **merge request** | is this the mechanism we want? | the design question no gate can answer |

Skipping the first is the common failure, and it is always justified the same
way: the fix is obvious.

## When to use this

On every patch for a confirmed finding, whether an agent or a person wrote it,
before the merge request exists. Also when reviewing a remediation workflow that
opens merge requests directly from a scanner going quiet.

## Procedure

**1 — Re-run the exploit against the patched build, in the replica.** Not the
scanner. The exploit is the only acceptance test that cannot be satisfied by
editing the code around a detector. A patch that leaves the exploit working has
failed regardless of what the scan says.

**2 — Prove the exploit still works against the *unpatched* build.** Same run,
same harness. Without this you cannot distinguish "the fix worked" from "the
probe broke", and a probe that silently stopped asserting reports every patch
as a success.

**3 — Run the regression suite in QA, against real-shaped data.** Behaviour
preserved is a separate claim from vulnerability closed, and the patch that
closes a SQL injection by returning an empty list satisfies the first gate
perfectly.

**4 — Diff the findings, both directions.** The target finding must be gone and
nothing new may appear. A patch that closes one injection and opens a path
traversal has a net score of zero and a merge request that says "fixes CVE".

**5 — Only now open the merge request, and put the evidence in it.** The
exploit run before and after, the regression result, the finding diff. A
reviewer's remaining job is the question no gate can answer — whether this is
the mechanism the codebase should use — and they can only get to it if the
first three are already answered on the page.

## Output contract

```json
{
  "candidates": [{"id": "str", "ci_green": true, "exploit_blocked": true,
                  "behaviour_preserved": true, "new_findings": 0,
                  "promoted_to": "rejected|qa|merge-request", "died_at": "str"}],
  "would_merge_on_ci_alone": 0,
  "merge_requests_opened": 0,
  "evidence_attached": ["str"]
}
```

`would_merge_on_ci_alone` against `merge_requests_opened` is the number this
procedure exists to produce. If they are equal, the gates changed nothing and
one of them is not actually running.

## Failure modes

- **Treating a green scan as proof.** It is proof the detector is quiet, and
  three of the four ways to quieten a detector are not fixes.
- **Skipping the replica because the fix is obvious.** The obvious fixes are
  where evasion hides, because nobody re-reads them.
- **Omitting the proof-of-fix run against the old build.** A broken probe then
  passes every patch.
- **Opening the merge request without the evidence.** The reviewer re-derives
  it, badly, or approves on the strength of the description.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/fix-promotion-gate/scripts/fix_promotion_gate.py
SCRIPT = "skills/appsec/fix-promotion-gate/scripts/fix_promotion_gate.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## 6 · The two patches that reach review, and why one of them still should not merge

Candidate C is the argument for the replica. It is green, the rule no longer
fires, and the exploit works unchanged — `escape()` moved the pattern past
the detector without closing the injection. Nothing but re-running the exploit
finds that, and nothing but an environment built to be attacked lets you.

Candidate D is the argument for the reviewer. It passes every gate: the exploit
is blocked, the regression suite is green, no new findings. It closes the bug by
returning an empty list, deleting partial-reference search — a feature
travellers use. No automated gate rejects it, and none should, because *is this
the mechanism we want* is not a property of a test run.

That is the division of labour the promotion path buys. The gates answer whether
the bug is closed, on the page, before the request is opened. The reviewer's
remaining job is D's question, which is the only one they were ever needed
for.

## What you just proved

The vulnerable build passes all four behaviour cases and the exploit returns 3 rows. Candidates A, B and D make the scanner green. Validation rejects B for changed behaviour and C for remaining exploitable, accepting A and D. Proof of fix holds for both accepted patches — the exploit works on the old build and fails on the new. The promotion path then walks the same four through the replica, QA and the finding diff: CI alone would have opened four merge requests and the gates open two, with C dying in the sandbox because the exploit still works and B dying in QA because the booking reference is silently truncated. D reaches review having passed every gate, which is the point — no automated check rejects a fix that works by deleting the feature.

## Your turn

Candidate D passes every automated gate and is still wrong. Write the rule that rejects it. You will find it has to be about which *mechanism* is acceptable, not about outcomes — and that rule belongs in your secure coding standard, not in the pipeline.

---

**Next → [B2.12 · Context engineering — cutting the false positives](https://spbreed.github.io/cyber-commons/lessons/B2.12.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.11.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.11.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*